# Phase 8 — Pipeline complet sur dataset personnel

## Objectif

Construire un pipeline complet sur un dataset réel :
- chargement ;
- préprocessing ;
- entraînement numpy from-scratch ;
- entraînement Keras ;
- comparaison des résultats.

Dataset choisi : Breast Cancer (classification binaire).

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

In [3]:
# ---- 1. Chargement ----
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Nom du dataset : Breast Cancer")

Shape X : (569, 30)
Shape y : (569,)
Classes : [0 1]
Nom du dataset : Breast Cancer


In [4]:
# ---- 2. Préprocessing ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train :", X_train.shape, y_train.shape)
print("Test  :", X_test.shape, y_test.shape)

Train : (455, 30) (455,)
Test  : (114, 30) (114,)


## 1. Pipeline numpy from-scratch

Architecture demandée :
- `n_features -> 16 -> 8 -> 1`
- initialisation He
- ReLU dans les couches cachées
- Sigmoid en sortie
- BCE loss
- 200 epochs

In [5]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [6]:
# ---- 3. Pipeline numpy from-scratch ----
np.random.seed(42)

n_features = X_train.shape[1]

W1 = np.random.randn(n_features, 16) * np.sqrt(2 / n_features)
b1 = np.zeros((1, 16))

W2 = np.random.randn(16, 8) * np.sqrt(2 / 16)
b2 = np.zeros((1, 8))

W3 = np.random.randn(8, 1) * np.sqrt(2 / 8)
b3 = np.zeros((1, 1))

lr = 0.01
n_epochs = 200
losses_numpy = []

start_numpy = time.time()

for epoch in range(n_epochs):
    # Forward
    z1 = X_train @ W1 + b1
    a1 = relu(z1)

    z2 = a1 @ W2 + b2
    a2 = relu(z2)

    z3 = a2 @ W3 + b3
    y_pred = sigmoid(z3).flatten()

    # Loss
    loss = bce_loss(y_train, y_pred)
    losses_numpy.append(loss)

    n = len(X_train)

    # Backward
    err3 = (y_pred - y_train).reshape(-1, 1)
    dW3 = (a2.T @ err3) / n
    db3 = np.sum(err3, axis=0, keepdims=True) / n

    err2 = (err3 @ W3.T) * relu_grad(z2)
    dW2 = (a1.T @ err2) / n
    db2 = np.sum(err2, axis=0, keepdims=True) / n

    err1 = (err2 @ W2.T) * relu_grad(z1)
    dW1 = (X_train.T @ err1) / n
    db1 = np.sum(err1, axis=0, keepdims=True) / n

    # Update
    W3 -= lr * dW3
    b3 -= lr * db3

    W2 -= lr * dW2
    b2 -= lr * db2

    W1 -= lr * dW1
    b1 -= lr * db1

numpy_train_time = time.time() - start_numpy

In [7]:
# Évaluation numpy sur le test set
a1_test = relu(X_test @ W1 + b1)
a2_test = relu(a1_test @ W2 + b2)
y_test_pred_numpy = sigmoid(a2_test @ W3 + b3).flatten()

numpy_test_acc = np.mean((y_test_pred_numpy > 0.5) == y_test)
numpy_final_loss = losses_numpy[-1]

print(f"Numpy from-scratch | Loss finale : {numpy_final_loss:.4f} | Test accuracy : {numpy_test_acc:.4f}")
print(f"Temps numpy : {numpy_train_time:.2f} s")

Numpy from-scratch | Loss finale : 0.2282 | Test accuracy : 0.8947
Temps numpy : 0.09 s


## 2. Pipeline Keras

Même architecture :
- Dense(16, relu)
- Dense(8, relu)
- Dense(1, sigmoid)

Avec :
- binary_crossentropy
- Adam lr=0.001
- 50 epochs
- batch_size=32
- validation_split=0.1

In [8]:
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(16, activation="relu", input_shape=(n_features,)),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

start_keras = time.time()

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

keras_train_time = time.time() - start_keras
keras_test_loss, keras_test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Keras | Loss finale : {keras_test_loss:.4f} | Test accuracy : {keras_test_acc:.4f}")
print(f"Temps Keras : {keras_train_time:.2f} s")

d:\Cours\5ème Année IPSSI - Paris\Machine Learning\deeplearning-j1-pmc-from-scratch-DONOU-Serge\.venv311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Keras | Loss finale : 0.0908 | Test accuracy : 0.9649
Temps Keras : 9.41 s


In [10]:
gain_points = (keras_test_acc - numpy_test_acc) * 100

print(f"Gain Keras vs Numpy : {gain_points:.1f} %")

Gain Keras vs Numpy : 7.0 %


In [11]:
print("\n=== TABLEAU COMPARATIF ===")
print(f"{'Modèle':18s} | {'Loss finale':12s} | {'Test accuracy':14s} | {'Temps (s)':10s}")
print("-" * 65)
print(f"{'Numpy from-scratch':18s} | {numpy_final_loss:.4f} | {numpy_test_acc:.4f} | {numpy_train_time:.2f}")
print(f"{'Keras':18s} | {keras_test_loss:.4f} | {keras_test_acc:.4f} | {keras_train_time:.2f}")


=== TABLEAU COMPARATIF ===
Modèle             | Loss finale  | Test accuracy  | Temps (s) 
-----------------------------------------------------------------
Numpy from-scratch | 0.2282 | 0.8947 | 0.09
Keras              | 0.0908 | 0.9649 | 9.41


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbes de loss
axes[0].plot(losses_numpy, label="Numpy train loss", linewidth=2)
axes[0].plot(history.history["loss"], label="Keras train loss", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Keras val loss", linewidth=2)
axes[0].set_title("Courbes de loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Barplot accuracy
models = ["Numpy", "Keras"]
accuracies = [numpy_test_acc, keras_test_acc]

axes[1].bar(models, accuracies)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_title("Accuracy comparée sur le test set")
axes[1].set_ylabel("Accuracy")

for i, v in enumerate(accuracies):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig("phase8_comparaison.png", dpi=100, bbox_inches="tight")
plt.show()

print("Figure sauvegardée : phase8_comparaison.png")

Figure sauvegardée : phase8_comparaison.png


C:\Users\serge\AppData\Local\Temp\ipykernel_26412\2343888452.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Analyse attendue

### Scénario normal
On doit obtenir :
- un pipeline complet numpy + Keras ;
- un tableau comparatif ;
- Keras légèrement meilleur que numpy en accuracy.

### Cas limite
Le support propose de tester des données manquantes ou des valeurs anormales, puis d'observer l'effet du traitement de ces valeurs.

### Scénario adversarial
Le support propose aussi d’ajouter une ligne extrême hors distribution dans le test set pour voir si le modèle fait une prédiction très confiante sur une donnée aberrante.

### Conclusion
Cette phase montre tout le pipeline :
- données ;
- préprocessing ;
- entraînement ;
- évaluation ;
- comparaison entre implémentation manuelle et framework haut niveau.